# Evaluation — Granite Speech

In [ ]:
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec

## Methods for Evaluating ASR

## Setup

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, device_map=device,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
).eval()

## Kaldi Test Data

In [ ]:
from pathlib import Path
import torchaudio
from IPython.display import Audio, display


dataset = []

dataset_path = Path("/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test") #other languages *_de *_fr *_es
# other datasets
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/eb/test *_de *_fr *_es
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/de/test *_de *_fr *_es 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en/test *_de *_fr *_es

wav_scp_dict = {}
with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
    for line in rf:
        id, wav_path = line.strip().split("\t")
        wav_scp_dict[id] = wav_path

text_dict = {}
with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
    for line in rf:
        id, text = line.strip().split("\t")
        text_dict[id] = text

for id, wav_path in wav_scp_dict.items():
    text = text_dict[id]
    dataset.append({"audio" : wav_path, "text" : text})

first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = torchaudio.load(first_entry["audio"])[0]
display(Audio(audio, rate=16000))

## Transcription Function

In [4]:
def transcribe(wav, prompt="can you transcribe the speech into a written format?", hotwords=None):
    if hotwords:
        prompt = f"{prompt}\nhotwords: {hotwords}"
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{prompt}"},
    ]
    text = processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = processor(text, wav, device=device, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1)
    return processor.tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:],
                                      skip_special_tokens=True).strip()

## Metrics (WER, CER, Error Counts)

In [5]:
import jiwer
import re
import string

def normalize(text):
    text = text.translate(str.maketrans('', '', string.punctuation)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text

## Run Inference on the Test Set

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

eval_rows = []
for sample in tqdm(dataset[:100]):
    model_output = transcribe(torchaudio.load(sample["audio"], normalize=True)[0])
    wer = jiwer.wer(sample["text"], model_output)
    cer = jiwer.cer(sample["text"], model_output)
    
    eval_rows.append({"audio_path": sample["audio"],
                      "label": normalize(sample["text"]),
                      "prediction": normalize(model_output),
                      "wer" : wer,
                      "cer": cer
                     }
                 )

results = pd.DataFrame(eval_rows)
results.head()

## Corpus-Level Results

In [ ]:
print("Mean WER:", results["wer"].mean())
print("Mean CER:", results["cer"].mean())
results[["wer", "cer"]].describe()

## Worst Samples

In [ ]:
results.sort_values("wer", ascending=False).head(10)


In [ ]:
from IPython.display import Audio, display

for _, row in results.sort_values("wer", ascending=False).head(10).iterrows():
    print("=" * 70)
    print("AUDIO  :", row["audio_path"])
    print("LABEL    :", row["label"])
    print("PRED    :", row["prediction"])
    print(f"WER {row["wer"]:.2%} | CER {row["cer"]:.2%}")
    display(Audio(torchaudio.load(row["audio_path"], normalize=True)[0], rate=16000))
